In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import datetime
import pickle

sys.path.insert(0, "cellular_cycle")

from cell_cycle_death_model import model
from model_fxns import *
from pysb.simulator import ScipyOdeSimulator
from pysb.simulator import BngSimulator


# Where to save large simulation output files
SAVE_DIR = Path(Path.home(),'ECM_mod_output')
SAVE_DIR.mkdir(exist_ok=True)

OVERWRITE = False # Overwrite existing simulation files

CURDATE = datetime.datetime.now().strftime('%Y%m%d')

SET_JPARAM = True # Set the parameter values for the simulation using PSO-optimized values
PSO_NUM = 2

### Define timesteps

In [13]:
t = np.linspace(0, 20000, 200001)

### Load model and update parameters

In [14]:
# Importing the parameters:
new_param = pd.read_csv('model_data/parameters.csv')
for key in new_param.keys():
    try:
        if model.parameters[key].value != new_param[key][0]:
            print(f'Changing {key} from {model.parameters[key].value} to {new_param[key][0]}')
            model.parameters[key].value = new_param[key]
    except:
        print(f'{key} not found in model parameters')

Changing kf1 from 0.01015115856168058 to 0.01
Changing kf2 from 2.4721161862856738 to 2.4
Changing kb from 0.1457584997626725 to 0.14
Changing kasbaxmbcl2 from 85.85921285402516 to 75
Changing kdsbaxmbcl2 from 0.075473443507626 to 0.05
Changing kasbh3bcl2 from 22.481505830327073 to 25
Changing kdsbh3bcl2 from 0.00940899371827974 to 0.01
Changing k1sbh3 from 0.001167462376821487 to 0.001
Changing k2sbh3 from 0.024640952836595526 to 0.025
Changing kdbh3 from 0.031067213881793124 to 0.03
Changing K_2E2F from 0.5901607221368979 to 0.5
Changing V_2E2F from 0.5713485451381991 to 0.7
Changing k_M from 0.20168774604603043 to 0.27
Changing k_EMY from 0.41091364947249037 to 0.37
Changing k_CDS from 0.25847119558573467 to 0.25
Changing k_ED from 0.21581187554741843 to 0.2
Changing K_p21fak from 0.6838167451240581 to 0.72
Changing F_n from 7.749090673634013 to 8
Changing k_E from 0.41292750251024796 to 0.4
Changing k_RE from 150.04993947946207 to 180
Changing K_S from 0.8589429536775087 to 0.8
Cha

In [15]:
if SET_JPARAM:
    with open('fit_data/pso/frac_prolif_parameters_pso_sorted.pkl', 'rb') as f:
        frac_prolif_parameters_pso = pickle.load(f)
    # new_param_pso_sort = frac_prolif_parameters_pso.sort_values([('cost', 'total cost likelihood')])
    # frac_prolif_parameters_pso[('cost', 'custom cost populations')] = frac_prolif_parameters_pso[('cost', 'cost population dynamics 0.6 kPa DMSO')] + frac_prolif_parameters_pso[('cost', 'cost population dynamics 4 kPa erlotinib')] + frac_prolif_parameters_pso[('cost', 'cost population dynamics 55 kPa erlotinib')]
    # new_param_pso_sort = frac_prolif_parameters_pso.sort_values(by=[('cost', 'custom cost populations')])
    new_params = frac_prolif_parameters_pso['model parameters']
    # Importing the parameters:
    PSO_num = PSO_NUM
    for key in new_params.keys():
        try:
            if model.parameters[key].value != new_params[key].iloc[PSO_num]:
                model.parameters[key].value = new_params[key].iloc[PSO_num]
        except:
            print(key)
    model.parameters['omega'].value = 200

In [16]:
if SET_JPARAM:
    filenames = [Path(SAVE_DIR,f) for f in os.listdir(SAVE_DIR) if 'stoch_sim' in f and '_psoparam_cycdi_0_0' in f]
else:
    filenames = [Path(SAVE_DIR,f) for f in os.listdir(SAVE_DIR) if 'stoch_sim' in f and '_param_cycdi_0_0' in f]
filenames = sorted(filenames)

file_order = (2,3,0,1,4,5)

stoch_filenames = [filenames[i] for i in file_order]
stoch_filenames

[WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_soft_DMSO_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_soft_erl_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_int_DMSO_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_int_erl_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_stiff_DMSO_psoparam_cycdi_0_0.csv'),
 WindowsPath('C:/Users/olesk/ECM_mod_output/stoch_sim_stiff_erl_psoparam_cycdi_0_0.csv')]

### Define stiffness values

In [17]:
vals = [0, 29, 49, 74, 99]

### Hardcoding AKT, ERK and FAK input values for erlotinib
Scaling values between low and high stiffness in 100 steps; corresponding stiffness values = 0-99.

In [18]:
treatment = 'ERL'
FAKs = list(np.linspace(0.4,0.6,50))
FAKs.extend(list(np.linspace(0.6,1,50)))
AKTs = list(np.linspace(0.3,0.6,50))
AKTs.extend([0.6]*50)
ERKs = [0.2]*100

### Run simulations and save output

In [19]:
for val in vals:
    model.parameters['FAK'].value = FAKs[val]
    model.parameters['AKT'].value = AKTs[val]
    model.parameters['ERK'].value = ERKs[val]
    
    num = f'{val:03}'
    
    simres = ScipyOdeSimulator(model, t, verbose=False, compiler='python').run()#deterministic simulation
    yout = simres.observables
    if SET_JPARAM:
        ode_filename = Path(SAVE_DIR, f'ODE_sim_{treatment}_stiff_{num}_pso_{PSO_num}.csv')
    else:
        ode_filename = Path(SAVE_DIR, f'ODE_sim_{treatment}_stiff_{num}_orig.csv')
    if ode_filename.exists():
        print(f"ODE simulation already exists at {ode_filename}")
    if not ode_filename.exists() or OVERWRITE:
        np.savetxt(ode_filename, yout, delimiter=',')
        print(f"ODE simulation saved to {ode_filename}")
    
    sim = BngSimulator(model, tspan=t, verbose=False) #stochastic simulator
    simres = sim.run(method='ssa', verbose=False, seed=33) #running the simulation; remove seed for truly random; seed=33 is used for similarity to deterministic sim
    yout = simres.observables
    if SET_JPARAM:
        stoch_filename = Path(SAVE_DIR, f'stoch_sim_{treatment}_stiff_{num}_pso_{PSO_num}.csv')
    else:
        stoch_filename = Path(SAVE_DIR, f'stoch_sim_{treatment}_stiff_{num}_orig.csv')
    if stoch_filename.exists():
        print(f"Stochastic simulation already exists at {stoch_filename}")
    if not stoch_filename.exists() or OVERWRITE:
        np.savetxt(stoch_filename, yout, delimiter=',')
        print(f"Stochastic simulation saved to {stoch_filename}")

ODE simulation already exists at C:\Users\olesk\ECM_mod_output\ODE_sim_ERL_stiff_000_pso_2.csv
ODE simulation saved to C:\Users\olesk\ECM_mod_output\ODE_sim_ERL_stiff_000_pso_2.csv
Stochastic simulation already exists at C:\Users\olesk\ECM_mod_output\stoch_sim_ERL_stiff_000_pso_2.csv
Stochastic simulation saved to C:\Users\olesk\ECM_mod_output\stoch_sim_ERL_stiff_000_pso_2.csv
ODE simulation already exists at C:\Users\olesk\ECM_mod_output\ODE_sim_ERL_stiff_029_pso_2.csv
ODE simulation saved to C:\Users\olesk\ECM_mod_output\ODE_sim_ERL_stiff_029_pso_2.csv
Stochastic simulation already exists at C:\Users\olesk\ECM_mod_output\stoch_sim_ERL_stiff_029_pso_2.csv
Stochastic simulation saved to C:\Users\olesk\ECM_mod_output\stoch_sim_ERL_stiff_029_pso_2.csv
ODE simulation already exists at C:\Users\olesk\ECM_mod_output\ODE_sim_ERL_stiff_049_pso_2.csv
ODE simulation saved to C:\Users\olesk\ECM_mod_output\ODE_sim_ERL_stiff_049_pso_2.csv
Stochastic simulation already exists at C:\Users\olesk\ECM_

### Hardcoding AKT, ERK and FAK input values for DMSO
Scaling values between low and high stiffness in 100 steps; corresponding stiffness values = 0-99.

In [20]:
treatment = 'DMSO'
FAKs = list(np.linspace(0.4,0.6,50))
FAKs.extend(list(np.linspace(0.6,1,50)))
AKTs = list(np.linspace(0.6,1,50))
AKTs.extend([1]*50)
ERKs = [1]*100

### Run simulations and save output

In [21]:
for val in vals:
    model.parameters['FAK'].value = FAKs[val]
    model.parameters['AKT'].value = AKTs[val]
    model.parameters['ERK'].value = ERKs[val]
    
    num = f'{val:03}'
    
    simres = ScipyOdeSimulator(model, t, verbose=False, compiler='python').run()#deterministic simulation
    yout = simres.observables
    if SET_JPARAM:
        ode_filename = Path(SAVE_DIR, f'ODE_sim_{treatment}_stiff_{num}_pso_{PSO_num}.csv')
    else:
        ode_filename = Path(SAVE_DIR, f'ODE_sim_{treatment}_stiff_{num}_orig.csv')
    if ode_filename.exists():
        print(f"ODE simulation already exists at {ode_filename}")
    if not ode_filename.exists() or OVERWRITE:
        np.savetxt(ode_filename, yout, delimiter=',')
        print(f"ODE simulation saved to {ode_filename}")
    
    sim = BngSimulator(model, tspan=t, verbose=False) #stochastic simulator
    simres = sim.run(method='ssa', verbose=False, seed=33) #running the simulation; remove seed for truly random; seed=33 is used for similarity to deterministic sim
    yout = simres.observables
    if SET_JPARAM:
        stoch_filename = Path(SAVE_DIR, f'stoch_sim_{treatment}_stiff_{num}_pso_{PSO_num}.csv')
    else:
        stoch_filename = Path(SAVE_DIR, f'stoch_sim_{treatment}_stiff_{num}_orig.csv')
    if stoch_filename.exists():
        print(f"Stochastic simulation already exists at {stoch_filename}")
    if not stoch_filename.exists() or OVERWRITE:
        np.savetxt(stoch_filename, yout, delimiter=',')
        print(f"Stochastic simulation saved to {stoch_filename}")



ODE simulation already exists at C:\Users\olesk\ECM_mod_output\ODE_sim_DMSO_stiff_000_pso_2.csv
ODE simulation saved to C:\Users\olesk\ECM_mod_output\ODE_sim_DMSO_stiff_000_pso_2.csv
Stochastic simulation already exists at C:\Users\olesk\ECM_mod_output\stoch_sim_DMSO_stiff_000_pso_2.csv
Stochastic simulation saved to C:\Users\olesk\ECM_mod_output\stoch_sim_DMSO_stiff_000_pso_2.csv
ODE simulation already exists at C:\Users\olesk\ECM_mod_output\ODE_sim_DMSO_stiff_029_pso_2.csv
ODE simulation saved to C:\Users\olesk\ECM_mod_output\ODE_sim_DMSO_stiff_029_pso_2.csv
Stochastic simulation already exists at C:\Users\olesk\ECM_mod_output\stoch_sim_DMSO_stiff_029_pso_2.csv
Stochastic simulation saved to C:\Users\olesk\ECM_mod_output\stoch_sim_DMSO_stiff_029_pso_2.csv
ODE simulation already exists at C:\Users\olesk\ECM_mod_output\ODE_sim_DMSO_stiff_049_pso_2.csv
ODE simulation saved to C:\Users\olesk\ECM_mod_output\ODE_sim_DMSO_stiff_049_pso_2.csv
Stochastic simulation already exists at C:\Users\